
# SAMPO2 Full Pipeline Walkthrough
This notebook allows you to execute the entire Trading System architecture step-by-step.
Each cell represents a major component of the data processing and training flow.


In [ ]:

# 1. Environment Setup
import sys
import os
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = str(Path(os.getcwd()).parent.parent) if 'sampo2' in os.getcwd() else os.getcwd()
if project_root not in sys.path:
    sys.path.append(project_root)

from sampo2.config import OUTPUT_DIR, COMBINED_DATA_FILE
print(f"Project Root: {project_root}")
print(f"Output Dir: {OUTPUT_DIR}")


## 2. Multi-Timeframe Data Aggregation
Aggregates Raw Data into H1, Daily, and M15 features.

In [ ]:

from sampo2.pipeline.system_architecture_pipeline import step_1_aggregate_data

# Run Aggregation (or load cached)
df_h1, df_d, df_m15 = step_1_aggregate_data()

print(f"H1 Shape: {df_h1.shape}")
print(f"Daily Shape: {df_d.shape}")
print(f"M15 Shape: {df_m15.shape}")


## 3. GARCH Volatility Modeling
Adds 'volatility_h1' feature.

In [ ]:

from sampo2.pipeline.system_architecture_pipeline import step_2_garch_volatility

df_h1 = step_2_garch_volatility(df_h1)
print(f"H1 Shape with Volatility: {df_h1.shape}")


## 4. Parallel Neural Network (PNN)
Trains a 3-Branch Network (H1+D+M15) and extracts Market Context embeddings.

In [ ]:

from sampo2.pipeline.system_architecture_pipeline import step_3_train_pnn, step_4_processor_merge

# Train PNN & Extract Embeddings
emb_df = step_3_train_pnn(df_h1, df_d, df_m15)

# Merge everything into Final Dataset
final_path = step_4_processor_merge(df_h1, emb_df)
print(f"Final Dataset saved to: {final_path}")


## 5. Renko Trend Prediction
Trains XGBoost on Renko Bricks and Enriches the Final Dataset with Predictions.

In [ ]:

from sampo2.data.renko_extractor import generate_renko_dataset
from sampo2.models.train_renko_xgboost import train_predictor, predict_full_dataset

# 1. Generate Bricks
generate_renko_dataset(input_file=final_path, block_size=0.0005)

# 2. Train Model
train_predictor()

# 3. Enrich H1 Dataset with Predictions
df_enriched = predict_full_dataset(final_path)
df_enriched.to_csv(final_path, index=True)
print("Dataset Enriched with Renko Signals.")


## 6. Hyperparameter Optimization (Optuna)
Optimizes the PPO Agent settings.

In [ ]:

from sampo2.agents.optuna_tuner import run_tuning

# Run 2 Trials for demonstration (Change to 5+ for real tuning)
run_tuning(n_trials=2)


## 7. PPO Agent Training
Trains the Final Agent using optimized parameters.

In [ ]:

from sampo2.agents.trainer import PPOTrainer

trainer = PPOTrainer()
# Train for 10,000 steps (Change to 50,000+ for full training)
trainer.train(total_timesteps=10000)


## 8. Evaluation
Simulates trading on unseen data.

In [ ]:

equity_curve = trainer.evaluate()

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(equity_curve)
plt.title("Agent Equity Curve (Validation)")
plt.xlabel("Steps")
plt.ylabel("Account Balance")
plt.show()
